In [1]:
import pandas as pd

In [2]:
df_clean = pd.read_csv("../data/student-mat-clean.csv")

In [3]:
df_clean.shape

(357, 33)

G1 and G2 are highly correlated with each other ,including G1,G2 the model will not learn from the data rather momorise the G1,G2
so we are removing the 2 columns now 

In [4]:
X = df_clean.drop(columns=["G1", "G2", "G3"])
y = df_clean["G3"]
print(X.shape, y.shape)

(357, 30) (357,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (285, 30)  Test: (72, 30)


In [6]:
cat_cols = X_train.select_dtypes(include="str").columns.tolist()
num_cols = X_train.select_dtypes(exclude="str").columns.tolist()
print(len(cat_cols), "categorical;", len(num_cols), "numeric")

17 categorical; 13 numeric


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

- the model can only work with numbers ,here there are lots of categorical columns so we encode it using onhot encoder and convert it to numbers .
one hot encoding is used because there are no order in these columns 
- standard scaler is used because the outlier values are so extreme ,so we scale these numbers 


In [8]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)
print(X_train_prep.shape, X_test_prep.shape)

(285, 43) (72, 43)


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

In [10]:
baseline_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LinearRegression()),
])

scores = cross_validate(baseline_pipe, X_train, y_train, cv=cv, scoring=scoring)

baseline_results = {
    "MAE": -scores["test_mae"].mean(),
    "RMSE": -scores["test_rmse"].mean(),
    "R2": scores["test_r2"].mean(),
}
pd.DataFrame(baseline_results, index=["Linear regression"]).round(2)

,MAE,RMSE,R2
Linear regression,2.44,3.07,0.09


- MAE 2.44: the model’s prediction is off by about 2.4 marks. For a student predicted at 12, the real grade is typically somewhere between 10 and 14.
- RMSE 3.07: a few predictions miss by more than mae
- R² 0.09: the model explains only about 9% of the differences in grades.
- this is the base , every new model must beat this score
- the reason r2 is low is becsuse we removed G1 ,G2 . so there is low data

In [11]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [14]:
from xgboost import XGBRegressor

models = {
    "Linear regression": LinearRegression(),
    "Decision tree": DecisionTreeRegressor(max_depth=4, random_state=42),
    "Random forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05,
                            max_depth=3, random_state=42),
}

In [15]:
results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    results[name] = {
        "MAE": -scores["test_mae"].mean(),
        "RMSE": -scores["test_rmse"].mean(),
        "R2": scores["test_r2"].mean(),
    }

comparison = pd.DataFrame(results).T.round(2)
comparison

,MAE,RMSE,R2
Linear regression,2.44,3.07,0.09
Decision tree,2.84,3.50,-0.18
Random forest,2.44,3.02,0.12
XGBoost,2.54,3.11,0.08


here when we compare other models with our linear regression model 
- decision tree is not good - it has high MAE and RMSE and its R2 is ne+gative so its worse in predicting
- random forest is better than linear ragression  it has low MAE , RMSE values and high R2 value
- XGBOOST is also not bad ,but random forest is better than xgboost 